In [3]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, sum, round

# Membuat SparkSession
spark = SparkSession.builder \
    .appName("Tugas6_ETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/30 12:56:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


In [4]:
# A. Extract
#Membaca data transaksi
df_transaksi = spark.read.csv(
    "tugas6_transaksi.csv",
    header=True,
    inferSchema=True
)

#Membaca data produk
df_produk = spark.read.json("tugas6_produk.json")

#Membaca data ulasan
df_ulasan = spark.read.csv(
    "tugas6_ulasan.csv",
    header=True,
    inferSchema=True
)

#Menampilkan jumlah baris dan schema
print("=== DATA TRANSAKSI ===")
print(f"Jumlah baris: {df_transaksi.count()}")
df_transaksi.printSchema()

print("=== DATA PRODUK ===")
print(f"Jumlah baris: {df_produk.count()}")
df_produk.printSchema()

print("=== DATA ULASAN ===")
print(f"Jumlah baris: {df_ulasan.count()}")
df_ulasan.printSchema()

=== DATA TRANSAKSI ===


Jumlah baris: 5000
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

=== DATA PRODUK ===
Jumlah baris: 30
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

=== DATA ULASAN ===
Jumlah baris: 3500
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [5]:
# B. Transform - Penggabungan
#Join transaksi dengan ulasan
#Menggunakan left join karena tidak semua transaksi memiliki ulasan
df_join = df_transaksi.join(
    df_ulasan,
    on="order_id",
    how="left"
)

#Join hasil sebelumnya dengan data produk
#Menggunakan inner join karena setiap transaksi memiliki produk yang valid
df_join = df_join.join(
    df_produk,
    on="product_id",
    how="inner"
)

#Menambahkan kolom total_pendapatan
df_join = df_join.withColumn(
    "total_pendapatan",
    col("unit_terjual") * col("harga")
)

#Menampilkan hasil
df_join.show(10)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|         1500000|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|          450000|
|        10|     TX5|           3|2026-10-01 00:00:00|     5| 75000|  Elektronik|  Produk-10|          225000|
|

In [6]:
# C. Transform
#Menambahkan kolom ada_ulasan sebelum mengisi nilai null
df_join = df_join.withColumn(
    "ada_ulasan",
    col("rating").isNotNull()
)

#Mengisi rating yang kosong dengan 0
#Nilai 0 digunakan sebagai penanda bahwa transaksi belum memiliki ulasan, karena rating yang valid hanya berada pada rentang 1 sampai 5
df_join = df_join.na.fill({
    "rating": 0
})

#Menampilkan hasil
df_join.show(10)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|ada_ulasan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|      true|
|         8|     TX1|           6|2026-10-25 00:00:00|     0|250000|     Fashion|   Produk-8|         1500000|     false|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|      true|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|      true|
|        19|     TX4|           6|2026-10-07 00:00:00|     0| 75000|Rumah Tangga|  Produk-19|          450000|     false|
|        10|     TX5|   

In [7]:
# D. Load
#Menentukan path penyimpanan di HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas6/hasil_etl

#Menyimpan hasil ETL dalam format Parquet
#Data dipartisi berdasarkan kategori
df_join.write \
    .mode("overwrite") \
    .partitionBy("kategori") \
    .parquet("hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl")

#Verifikasi struktur folder di HDFS
!hdfs dfs -ls -R /user/mahasiswa/tugas6/hasil_etl

#Membaca kembali data hasil ETL dari HDFS
df_hasil = spark.read.parquet("hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl")

#Menampilkan jumlah baris sebagai bukti data tersimpan
print(f"Jumlah baris hasil ETL: {df_hasil.count()}")

-rw-r--r--   3 putrisenja supergroup          0 2026-09-30 13:00 /user/mahasiswa/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - putrisenja supergroup          0 2026-09-30 13:00 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 putrisenja supergroup      16363 2026-09-30 13:00 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik/part-00000-720df7b3-6da3-49df-add2-8e41873fc8ad.c000.snappy.parquet
drwxr-xr-x   - putrisenja supergroup          0 2026-09-30 13:00 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 putrisenja supergroup      11130 2026-09-30 13:00 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion/part-00000-720df7b3-6da3-49df-add2-8e41873fc8ad.c000.snappy.parquet
drwxr-xr-x   - putrisenja supergroup          0 2026-09-30 13:00 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 putrisenja supergroup      10388 2026-09-30 13:00 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan/part-00000-720df7b3-6da3-49df-add2-8e41873fc8ad.c000.snap

In [8]:
# E. Insight Akhir
#Menghitung total transaksi dan transaksi yang memiliki ulasan
df_insight = df_hasil.groupBy("kategori").agg(
    count("*").alias("total_transaksi"),
    sum(
        when(col("ada_ulasan") == True, 1).otherwise(0)
    ).alias("transaksi_dengan_ulasan")
)

#Menghitung persentase transaksi yang memiliki ulasan
df_insight = df_insight.withColumn(
    "persentase_ulasan",
    round(
        col("transaksi_dengan_ulasan") /
        col("total_transaksi") * 100,
        2
    )
)

#Mengurutkan dari persentase ulasan paling rendah
df_insight = df_insight.orderBy("persentase_ulasan")

#Menampilkan hasil
df_insight.show()

#Menampilkan kategori dengan persentase ulasan paling rendah
kategori_terendah = df_insight.select(
    "kategori",
    "persentase_ulasan"
).first()

print(
    f"Kategori dengan persentase transaksi dengan ulasan paling rendah: "
    f"{kategori_terendah['kategori']} "
    f"({kategori_terendah['persentase_ulasan']}%)"
)

+------------+---------------+-----------------------+-----------------+
|    kategori|total_transaksi|transaksi_dengan_ulasan|persentase_ulasan|
+------------+---------------+-----------------------+-----------------+
|     Makanan|            536|                    369|            68.84|
|   Kesehatan|            961|                    662|            68.89|
|     Fashion|           1035|                    726|            70.14|
|Rumah Tangga|            815|                    575|            70.55|
|  Elektronik|           1653|                   1168|            70.66|
+------------+---------------+-----------------------+-----------------+

Kategori dengan persentase transaksi dengan ulasan paling rendah: Makanan (68.84%)


In [9]:
#Menutup SparkSession setelah selesai
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
